In [17]:
import pyexasol
import configparser
import pandas as pd
from datetime import date
import re

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\pfxPROD.ini')

dsn=config['pfxPROD']['dsn']
user=config['pfxPROD']['user']
pwd=config['pfxPROD']['pwd']
schema=config['pfxPROD']['schema']
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

In [33]:
### LOCATION OF THE FILE
path = 'C:\\Users\\svi02\\Documents\\HPO\HS_Files\\Archiv\\'
# filename = 'GREYLIST_(2018-07-16).xlsx'
filename = 'BLACKLIST_(2019-09-27).xlsx'
string = re.split('_', filename)[0]
df = pd.read_excel (path+filename)
df.rename(columns={'Blacklist or Graylist': 'FILTER_STATUS', 'Hkey' : 'HOTEL_ID'}, errors='raise', inplace=True)
df['FILTER_LOAD_DATE'] = date.today()

df = df[df['FILTER_STATUS'].str.upper()==('black' if string == 'BLACKLIST' else 'gray').upper()]
df['FILTER_STATUS'] = string

connect.execute(f"DELETE FROM DWHPFX.FILTER_STATUS_TEST WHERE FILTER_STATUS = '{string}'")
connect.import_from_pandas(df[['HOTEL_ID', 'FILTER_STATUS', 'FILTER_LOAD_DATE']], ('DWHPFX', 'FILTER_STATUS_TEST'))

In [38]:
test = {  
   "Records":[  
      {  
         "eventVersion":"2.2",
         "eventSource":"aws:s3",
         "awsRegion":"us-west-2",
         "eventTime":"The time, in ISO-8601 format, for example, 1970-01-01T00:00:00.000Z, when Amazon S3 finished processing the request",
         "eventName":"event-type",
         "userIdentity":{  
            "principalId":"Amazon-customer-ID-of-the-user-who-caused-the-event"
         },
         "requestParameters":{  
            "sourceIPAddress":"ip-address-where-request-came-from"
         },
         "responseElements":{  
            "x-amz-request-id":"Amazon S3 generated request ID",
            "x-amz-id-2":"Amazon S3 host that processed the request"
         },
         "s3":{  
            "s3SchemaVersion":"1.0",
            "configurationId":"ID found in the bucket notification configuration",
            "bucket":{  
               "name":"bucket-name",
               "ownerIdentity":{  
                  "principalId":"Amazon-customer-ID-of-the-bucket-owner"
               },
               "arn":"bucket-ARN"
            },
            "object":{  
               "key":"object-key",
               "size":"object-size",
               "eTag":"object eTag",
               "versionId":"object version if bucket is versioning-enabled, otherwise null",
               "sequencer": "a string representation of a hexadecimal value used to determine event sequence, only used with PUTs and DELETEs"
            }
         },
         "glacierEventData": {
            "restoreEventData": {
               "lifecycleRestorationExpiryTime": "The time, in ISO-8601 format, for example, 1970-01-01T00:00:00.000Z, of Restore Expiry",
               "lifecycleRestoreStorageClass": "Source storage class for restore"
            }
         }
      }
   ]
}

print(test['Records'][0]['s3']['bucket']['name'])

bucket-name


In [40]:
print(test['Records'][0]['s3']['object']['key'])

object-key


In [41]:
df

,HOTEL_ID,Hotel Name,Chain ID,Chain Name,HS Office,FILTER_STATUS,DCA Case,Reason,FILTER_LOAD_DATE
0,73,Hotel Lochmühle,0.0,Individualhotel,HS Cologne,BLACKLIST,Y 12.11.2018,DCA List. 12.11.2018,2021-04-27
1,353,Atlantis,0.0,Individualhotel,HS Cologne,BLACKLIST,Y 12.11.2018,DCA List. 12.11.2018,2021-04-27
2,363,Sheraton,15.0,Marriott International,HS Rome,BLACKLIST,Y 12.11.2018,DCA List. 12.11.2018,2021-04-27
3,1036,Panorama Hotel Schloßberg,0.0,Individualhotel,HS Cologne,BLACKLIST,Y 12.11.2018,DCA List. 12.11.2018,2021-04-27
4,1201,Parkhotel,0.0,Individualhotel,HS Cologne,BLACKLIST,N,Britta Jansen. Hotel is not interested in any ...,2021-04-27
...,...,...,...,...,...,...,...,...,...
1262,154444,Mamaison Hotel Le Regina Warsaw,1603.0,"CPI Hotels, a.s.",HS Warsaw,BLACKLIST,N,hotels rejected definitely all incremental rfps,2021-04-27
1263,216331,Mamaison Residence Diana Warsaw,1603.0,"CPI Hotels, a.s.",HS Warsaw,BLACKLIST,N,hotels rejected definitely all incremental rfps,2021-04-27
1264,391454,St. George Residence All Suite Hotel DeLuxe,0.0,Individualhotel,HS Warsaw,BLACKLIST,N,hotels rejected definitely all incremental rfps,2021-04-27
1265,421394,Hornigold Euroresidence,0.0,Individualhotel,HS Warsaw,BLACKLIST,N,hotels rejected definitely all incremental rfps,2021-04-27


In [45]:
for index, row in df.iterrows():
    print(row['HOTEL_ID'])

73
353
363
1036
1201
1347
1457
3112
3216
3252
3465
3476
3531
3732
4029
4041
4332
4343
4984
5144
5234
5623
6016
6570
6707
6708
6893
6950
7534
9545
9707
9813
10425
10933
11467
11808
11964
12130
12198
12385
12495
12522
12544
12607
12621
12889
13349
13442
13784
14115
14489
15418
15782
16129
16782
16835
16891
17108
17176
17283
17391
18765
19007
19163
19297
19650
19655
19781
20109
20728
22539
22540
22635
22798
23537
24069
25168
25274
25581
26186
26233
26434
27451
27712
28267
28283
28891
29971
30596
30966
31126
31701
33360
33617
33834
34358
34453
34872
35030
35091
35666
35787
35877
36033
36506
37199
37942
37971
38167
38897
39100
39716
39942
39976
40085
40597
41762
41811
42245
42302
42326
42655
43274
43862
44335
44703
45319
45323
45591
45944
45975
45976
46054
46929
47005
47319
50272
50799
51289
51616
51947
52395
53150
55085
55236
55470
55574
55593
55864
55874
55900
55957
55987
56468
56865
57095
57307
60113
60277
60594
60651
60723
62353
62610
63088
63456
64250
64355
64478
64648
64798
64826
6484

In [7]:
import pandas as pd
from datetime import date
import re
### LOCATION OF THE FILE
path = 'C:\\Users\\svi02\\Documents\\HPO\\HS_Files\\Archiv\\'
# filename = 'GREYLIST_(2018-07-16).xlsx'
filename = 'BLACKLIST_2019-09-27.xlsx'
string = re.split('_', filename)[0]
df = pd.read_excel(path+filename)
df.rename(columns={'Blacklist or Graylist': 'FILTER_STATUS', 'Hkey' : 'HOTEL_ID'}, errors='raise', inplace=True)
df['FILTER_LOAD_DATE'] = date.today()

# df = df[df['FILTER_STATUS'].str.upper()==('black' if string == 'BLACKLIST' else 'gray').upper()]
# df['FILTER_STATUS'] = string

In [9]:
df['FILTER_STATUS'].str.upper()==('black' if string == 'BLACKLIST' else 'gray').upper()

0       True
1       True
2       True
3       True
4       True
        ... 
1262    True
1263    True
1264    True
1265    True
1266    True
Name: FILTER_STATUS, Length: 1267, dtype: bool